[参考文档](https://code.claude.com/docs/en/agent-sdk/observability)

当你的 Agent 在生产环境中运行时，你需要清楚地知道：

* 它具体调用了哪些工具？
* 每次向大模型发送请求花了多长时间？
* 到底消耗了多少 Token？花了多少钱？
* 如果出错了，是在哪一步、因为什么崩溃的？

为了统一且高效地收集这些信息，Claude Agent SDK 采用了业界标准的 **OpenTelemetry (OTel)** 协议，将这些数据导出到你喜欢的监控后台（比如 Datadog、Grafana、Honeycomb,langfuse 等）。

---

## 1. 核心原理：数据是怎么产生的？

你需要先理解一个关键的架构设计：**Agent SDK 本身并不直接生成这些监控数据**。

SDK 在运行你的代码时，会在底层启动一个 `Claude Code CLI`（命令行工具）作为子进程，由它来与 Claude 的服务器通信。所有的监控数据（遥测数据）都是由底层的 CLI 默默记录的。

你只需要通过**环境变量 (Environment Variables)** 将配置传递给 CLI，它就会自动把数据发送给你的监控平台，整个过程完全不需要你修改核心的业务逻辑代码。

---

## 2. 可观测性的“三驾马车”

Claude 提供了三种独立的监控信号，你可以根据需求像拼图一样单独开启它们：

| 信号类型 (Signal) | 包含什么内容？ | 对应的开启开关 |
| --- | --- | --- |
| **Metrics (指标)** | 核心统计数据：Token 消耗量、花费成本、代码行数、工具调用次数等。 | `OTEL_METRICS_EXPORTER` |
| **Logs (日志)** | 结构化记录：系统发生的事件，比如 API 请求、报错信息、工具返回的结果。 | `OTEL_LOGS_EXPORTER` |
| **Traces (链路追踪)** | 运行时间轴：一次交互从开始到结束的完整流程图，包括模型请求、工具调用的层级关系和耗时（目前是 Beta 功能）。 | `OTEL_TRACES_EXPORTER` 配合 `CLAUDE_CODE_ENHANCED_TELEMETRY_BETA=1` |

---

## 3. 如何在代码中配置？(实战演示)

要让监控生效，你必须设置总开关 `CLAUDE_CODE_ENABLE_TELEMETRY=1`，并且至少配置一个导出器（Exporter，通常填 `otlp`）。

这里以 **Python** 为例，演示如何通过 `options.env` 在单次调用中配置（如果你用 TypeScript，原理是一样的，只需注意 TS 需要先解构合并 `process.env`）：

```python
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions

# 1. 定义 OpenTelemetry 的环境变量配置
OTEL_ENV = {
    "CLAUDE_CODE_ENABLE_TELEMETRY": "1",          # 开启监控的总开关
    "CLAUDE_CODE_ENHANCED_TELEMETRY_BETA": "1",   # 开启详细的链路追踪特性
    
    # 为三种信号选择导出方式 (otlp 代表通过标准网络协议发送)
    "OTEL_TRACES_EXPORTER": "otlp",
    "OTEL_METRICS_EXPORTER": "otlp",
    "OTEL_LOGS_EXPORTER": "otlp",
    
    # 配置接收数据的服务器地址和鉴权信息 (需替换成你的监控平台地址)
    "OTEL_EXPORTER_OTLP_PROTOCOL": "http/protobuf",
    "OTEL_EXPORTER_OTLP_ENDPOINT": "http://collector.example.com:4318",
    "OTEL_EXPORTER_OTLP_HEADERS": "Authorization=Bearer your-token",
}

async def main():
    # 2. 将环境配置打包传入 Agent 选项
    options = ClaudeAgentOptions(env=OTEL_ENV)
    
    # 3. 运行 Agent
    async for message in query(
        prompt="请列出当前目录下的所有文件",
        options=options
    ):
        print(message)

asyncio.run(main())

```

---

## 4. 新手避坑指南

在实际开发中，有几个非常重要的细节需要注意：

### 坑一：程序运行太快导致数据丢失

监控数据并不是产生一条就发一条，而是“攒到一定数量”再批量发送（指标默认攒 60 秒，日志默认 5 秒）。如果你的 Agent 执行个简单的任务，2秒就跑完了直接退出，还在缓存里的数据就会丢失。

* **解决办法：** 显式缩短发送间隔时间。
```python
"OTEL_METRIC_EXPORT_INTERVAL": "1000",  # 缩短到 1000 毫秒 (1秒)
"OTEL_LOGS_EXPORT_INTERVAL": "1000",
"OTEL_TRACES_EXPORT_INTERVAL": "1000",

```



### 坑二：数据隐私与敏感信息

由于 AI Agent 经常处理公司的核心代码或私密文件，默认情况下，SDK **只记录结构信息（如耗时、错误码、模型名字），绝不会记录你输入的 Prompt 或读取的文件内容**。
如果你确实在调试阶段需要看具体的输入输出内容，需要你**手动明确授权**（千万不要在非安全环境开启）：

* 记录用户的 Prompt：`OTEL_LOG_USER_PROMPTS=1`
* 记录工具输入参数：`OTEL_LOG_TOOL_DETAILS=1`
* 记录完整的 API 请求体：`OTEL_LOG_RAW_API_BODIES=1`

### 坑三：怎么区分是哪个用户调用的？

如果你开发了一个有很多用户的服务。你需要知道监控大盘里的某次报错到底是谁触发的。你可以利用 `OTEL_RESOURCE_ATTRIBUTES` 为数据打上标签：

```python
# 这样在监控后台，你就能精准搜索出 user_001 的所有运行轨迹
"OTEL_RESOURCE_ATTRIBUTES": "enduser.id=user_001,tenant.id=company_A"

```



代码示例待补充，课程没有使用langfuse, 我建议AI native项目直接用langfuse.